# Warcraft III — кои единици има на screenshot-а?

**Версия 2** на проекта: входът е **цял screenshot от играта** (класическа графика), а изходът е списък
с единиците на него, например *"Human Footman, Orc Grunt, Undead Ghoul"*.

Това е **multi-label класификация** — една снимка може да има няколко етикета. Тя е обяснена в
[fastbook, глава 6](https://github.com/fastai/fastbook/blob/master/06_multicat.ipynb) и се различава от
Lesson 1 само на три места:

| | Lesson 1 (един етикет) | Тук (много етикети) |
|---|---|---|
| блок за изхода | `CategoryBlock` | `MultiCategoryBlock` |
| последен слой | softmax (сборът е 100%) | sigmoid (всяка единица отделно 0–100%) |
| метрика | `error_rate` | `accuracy_multi`, `F1ScoreMulti` |

Моделът познава **48 единици** — всички обикновени единици на 4-те раси (без героите).

## 1. Данните

Снимките не са от интернет, а **направени от самата игра**. Картата в `game_capture/dataset_trigger.j`
слага 1–6 случайни единици на случайно място (случайни цветове, посоки, анимации, ден/нощ, понякога сграда),
а `game_capture/capture.py` прави screenshot и записва кои единици има.

Затова етикетите са **винаги верни** — не е нужно ръчно почистване като във версия 1.
Всяка карта е отделен файл с име `<карта>_<номер>.jpg`.

In [ ]:
from fastai.vision.all import *

df = pd.read_csv('data/labels.csv')
df['map'] = df['fname'].str.rsplit('_', n=1).str[0]
df.head()

In [ ]:
print(df['map'].value_counts())
df['labels'].str.split(';').explode().value_counts()

## 2. Честна проверка: validation от карта, която моделът не е виждал

Ако разделим снимките случайно, validation ще има същите карти като обучението и резултатът ще е
по-добър от реалния. Затова **цяла карта** остава само за проверка (както в fastbook се пази отделна
папка `valid`).

Имаме 4 карти: map1 и map4 са Lordaeron Summer (трева и пръст), map2 е Northrend (сняг и лед),
map3 е Ashenvale (гора). За проверка оставяме **map4**: такъв терен моделът е виждал (map1), но самата
карта — никога. Така при обучението има и от трите вида терен.

In [ ]:
valid_map = 'map4'   # Lordaeron Summer like map1, but a map the model never sees in training
df['is_valid'] = df['map'] == valid_map
valid_map

In [ ]:
dls = DataBlock(
    blocks=(ImageBlock, MultiCategoryBlock),
    get_x=ColReader('fname', pref='data/screens/'),
    get_y=ColReader('labels', label_delim=';'),
    splitter=ColSplitter('is_valid'),
    item_tfms=Resize((360, 640), method='squish'),
    batch_tfms=aug_transforms(max_rotate=0, max_zoom=1, max_warp=0)
).dataloaders(df, bs=16)

dls.show_batch(max_n=4, figsize=(16, 9))

Снимките са 640×360 (а не 224×224 като в Lesson 1) — единиците на цял екран са малки и при 224 пиксела
биха се превърнали в няколко точки. Augmentation-ът е само огледално обръщане и осветление:
без завъртане и zoom, за да не изрежем единица от края на снимката.

## 3. Обучение

In [ ]:
learn = vision_learner(dls, 'convnext_tiny_in22k',
                       metrics=[partial(accuracy_multi, thresh=0.5), F1ScoreMulti(average='macro')])
learn.fine_tune(6)

**Внимание:** `accuracy_multi` гледа всяка от 48-те единици поотделно. На снимка с 3 единици моделът може
да каже "няма я" за останалите 45 и пак да има 94% — затова гледаме и **F1** (колко от истинските
единици намира и колко от намерените са верни).

## 4. Какъв праг (threshold)?

Sigmoid дава число 0–1 за всяка единица. Над прага казваме "има я". Както в fastbook глава 6,
пробваме различни прагове върху validation set-а:

In [ ]:
from sklearn.metrics import f1_score

preds, targs = learn.get_preds()
xs = torch.linspace(0.05, 0.95, 19)
accs = [accuracy_multi(preds, targs, thresh=t, sigmoid=False) for t in xs]
f1s = [f1_score(targs, preds > t, average='macro', zero_division=0) for t in xs]
plt.plot(xs, accs, label='accuracy_multi'); plt.plot(xs, f1s, label='F1'); plt.legend(); plt.xlabel('threshold')
threshold = float(xs[int(np.argmax(f1s))])
print('best threshold:', threshold)

## 5. Кои единици познава добре и кои — не?

- **recall** — от всички пъти, когато единицата е на снимката, в колко % моделът я намира;
- **precision** — когато моделът каже "тук има X", в колко % е прав.

In [ ]:
found = preds > threshold
tp = (found & (targs == 1)).sum(0).float()
per_unit = pd.DataFrame({'unit': list(dls.vocab),
                         'recall': (tp / targs.sum(0)).numpy(),
                         'precision': (tp / found.sum(0).clamp(min=1)).numpy()}).round(2)
per_unit.sort_values('recall')

In [ ]:
learn.show_results(max_n=4, figsize=(16, 12))

## 6. Проба с истински screenshot

Ако сте направили screenshot-и от истинска игра (бутон **Print Screen** в Warcraft III ги записва в
`Documents\Warcraft III\Screenshots`), сложете ги в папка `my_screenshots/` и пуснете:

In [ ]:
for f in get_image_files('my_screenshots')[:5] if Path('my_screenshots').exists() else []:
    _, found_units, probs = learn.predict(f)
    print(f.name, [u for u, p in zip(dls.vocab, probs) if p > threshold])

## 7. Export за сайта (GitHub Pages)

Моделът работи директно в браузъра (вижте README):
- **ONNX** — общ формат, който JavaScript може да пусне с `onnxruntime-web`;
- **quantization** (int8) смалява файла около 3 пъти — GitHub не приема файлове над 100 MB;
- `docs/model.json` казва на страницата имената на единиците, размера на снимката и прага;
- няколко снимки от validation картата стават примери на сайта.

In [ ]:
import json, shutil
from onnxruntime.quantization import quantize_dynamic, QuantType

learn.export('model.pkl')
learn.model.eval().cpu()
torch.onnx.export(learn.model, torch.randn(1, 3, 360, 640), 'model.onnx',
                  input_names=['image'], output_names=['logits'], dynamo=False)
quantize_dynamic('model.onnx', 'docs/model.onnx', weight_type=QuantType.QInt8,
                 op_types_to_quantize=['MatMul', 'Gemm'], per_channel=True)

examples = df[df['map'] == valid_map]['fname'].head(8).tolist()
shutil.rmtree('docs/examples', ignore_errors=True)
Path('docs/examples').mkdir()
for f in examples: shutil.copy(f'data/screens/{f}', 'docs/examples')

Path('docs/model.json').write_text(json.dumps({
    'labels': list(dls.vocab), 'multi_label': True, 'threshold': threshold,
    'height': 360, 'width': 640, 'examples': examples}))

for f in ['model.pkl', 'model.onnx', 'docs/model.onnx']: print(f, Path(f).stat().st_size // 1_000_000, 'MB')